In [0]:
#Catalog 
catalog = "workspace"

#source Schema
source_schema = "silver"

#Source Object 
source_object = "silver_bookings"

#CDC Column 
cdc_column = "modifyDate"

#BCK- DATED REFRESH
bckdated_refresh = ""

# Source Fact Table
fact_table = f"{catalog}.{source_schema}.{source_object}"

#Target Schema 
target_schema = "gold"

#Target Object
target_object = "FactBookings"

In [0]:
dimensions = [
    {
        "table": f"{catalog}.{target_schema}.DimPassengers",
        "alias": "DimPassengers",
        "join_keys": [("passenger_id", "passenger_id")]  # (fact_col, dim_col)
    },
    {
        "table": f"{catalog}.{target_schema}.DimFlights",
        "alias": "DimFlights",
        "join_keys": [("flight_id", "flight_id")]  # (fact_col, dim_col)
    },
    {
       "table": f"{catalog}.{target_schema}.DimAirports",
        "alias": "DimAirports",
        "join_keys": [("airport_id", "airport_id")]  # (fact_col, dim_col)
    },   
]

# Columns you want to keep from Fact table (besides the surrogate keys)
fact_columns = ["amount", "booking_date", "modifyDate"]

**LAST LOAD DATE**

In [0]:
# No Back Dated Refresh
if len(bckdated_refresh) == 0:
# If the table Exists in the Destination
    if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):

        last_load = spark.sql(f"select max({cdc_column}) from workspace. {target_schema}.{target_object}").collect()[0][0]
    else:
        last_load = "1900-01-01 00:00:00"

# Yes Back Dated Refresh
else:
    last_load = bckdated_refresh

   
#Test the last load
last_load

'1900-01-01 00:00:00'

**DYNAMIC FACT QUERY [BRING KEYS}**

In [0]:
def generate_fact_query_incremental(fact_table, dimensions, fact_columns, cdc_column, last_load):
    fact_alias = "f"

    # Base columns to select (WITHOUT cdc_column)
    select_cols = [f"{fact_alias}.{col}" for col in fact_columns]

    # Build joins dynamically
    join_clauses = []
    for dim in dimensions:
        table_full = dim["table"]
        alias = dim["alias"]
        table_name = table_full.split('.')[-1]
        surrogate_key = f"{alias}.{table_name}Key"
        select_cols.append(surrogate_key)

        # Build ON clause
        on_conditions = [
            f"{fact_alias}.{fk} = {alias}.{dk}" for fk, dk in dim["join_keys"]
        ]
        join_clause = f"LEFT JOIN {table_full} {alias} ON " + " AND ".join(on_conditions)
        join_clauses.append(join_clause)

    # Final SELECT and JOIN clauses
    select_clause = ",\n        ".join(select_cols)
    joins = "\n".join(join_clauses)

    # WHERE clause for incremental filtering
    where_clause = f"{fact_alias}.{cdc_column} >= '{last_load}'"

    # Final query
    query = f"""
SELECT
        {select_clause}
FROM {fact_table} {fact_alias}
{joins}
WHERE {where_clause}
""".strip()

    return query

In [0]:
query = generate_fact_query_incremental(fact_table, dimensions, fact_columns, cdc_column, last_load)

## **DF_FACT** ##


In [0]:
print(query)

SELECT
        f.amount,
        f.booking_date,
        f.modifyDate,
        DimPassengers.DimPassengersKey,
        DimFlights.DimFlightsKey,
        DimAirports.DimAirportsKey
FROM workspace.silver.silver_bookings f
LEFT JOIN workspace.gold.DimPassengers DimPassengers ON f.passenger_id = DimPassengers.passenger_id
LEFT JOIN workspace.gold.DimFlights DimFlights ON f.flight_id = DimFlights.flight_id
LEFT JOIN workspace.gold.DimAirports DimAirports ON f.airport_id = DimAirports.airport_id
WHERE f.modifyDate >= '1900-01-01 00:00:00'


In [0]:
df_fact = spark.sql(query)

In [0]:
from pyspark.sql.functions import current_timestamp, lit

df_fact_final = df_fact.withColumn("load_date", current_timestamp())\
    .withColumn("processing_date", lit("2026-05-10"))

df_fact_final.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema", "true")\
    .saveAsTable(f"{catalog}.{target_schema}.{target_object}")

# Verify
spark.sql(f"SELECT * FROM {catalog}.{target_schema}.{target_object} LIMIT 10").display()

amount,booking_date,modifyDate,DimPassengersKey,DimFlightsKey,DimAirportsKey,load_date,processing_date
850.72,2025-05-29,2026-05-04T16:48:23.716Z,null,24,53,2026-05-11T01:23:48.003Z,2026-05-10
376.63,2025-06-09,2026-05-04T16:48:23.716Z,null,62,8,2026-05-11T01:23:48.003Z,2026-05-10
534.02,2025-06-03,2026-05-04T16:48:23.716Z,null,33,17,2026-05-11T01:23:48.003Z,2026-05-10
1333.7,2025-06-16,2026-05-04T16:48:23.716Z,null,11,44,2026-05-11T01:23:48.003Z,2026-05-10
1334.96,2025-06-17,2026-05-04T16:48:23.716Z,null,29,13,2026-05-11T01:23:48.003Z,2026-05-10
296.13,2025-05-18,2026-05-04T16:48:23.716Z,null,13,9,2026-05-11T01:23:48.003Z,2026-05-10
460.14,2025-04-05,2026-05-04T16:48:23.716Z,null,78,21,2026-05-11T01:23:48.003Z,2026-05-10
1402.02,2025-06-04,2026-05-04T16:48:23.716Z,null,58,22,2026-05-11T01:23:48.003Z,2026-05-10
1444.51,2025-05-16,2026-05-04T16:48:23.716Z,null,91,21,2026-05-11T01:23:48.003Z,2026-05-10
292.39,2025-05-16,2026-05-04T16:48:23.716Z,null,99,39,2026-05-11T01:23:48.003Z,2026-05-10
